# 📊 Notebook 6: Model Evaluation

This notebook provides comprehensive evaluation of all trained models:
- ROC-AUC curves
- Precision-Recall curves
- Confusion matrices
- Threshold optimization
- Feature importance (tree models)
- SHAP values (model interpretability)
- Business cost analysis per model


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle, os, warnings
from sklearn.metrics import (
    roc_auc_score, roc_curve, average_precision_score,
    precision_recall_curve, confusion_matrix,
    classification_report, f1_score, precision_score, recall_score
)
import sys
sys.path.insert(0, '..')
from src.evaluate import (
    evaluate_model, plot_roc_curves, plot_precision_recall_curves,
    plot_confusion_matrix, find_optimal_threshold, compute_business_roi
)
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#0f1117',
    'text.color': 'white', 'axes.labelcolor': 'white',
    'xtick.color': 'white', 'ytick.color': 'white',
    'axes.edgecolor': '#444',
})

# Load saved evaluation data
eval_data = pickle.load(open('../models/eval_data.pkl', 'rb'))
probs = eval_data['probs']
y_test = eval_data['y_test']
X_test = eval_data['X_test']
feature_cols = eval_data['feature_cols']

print(f"Test set: {len(y_test):,} rows | Fraud: {y_test.sum():,} ({y_test.mean()*100:.3f}%)")
print("Models loaded:", list(probs.keys()))


In [ ]:
# ── ROC Curves ────────────────────────────────────────────────────────────
fig = plot_roc_curves(probs, y_test, figsize=(10, 7),
                      save_path='../visuals/06_roc_curves.png')


In [ ]:
# ── Precision-Recall Curves ───────────────────────────────────────────────
fig = plot_precision_recall_curves(probs, y_test, figsize=(10, 7),
                                   save_path='../visuals/06_pr_curves.png')


In [ ]:
# ── Full Metrics Table ────────────────────────────────────────────────────
metrics_list = []
for name, y_prob in probs.items():
    opt_thresh = find_optimal_threshold(y_test, y_prob, metric='f1')
    m = evaluate_model(y_test, y_prob, model_name=name, threshold=opt_thresh)
    metrics_list.append(m)

metrics_df = pd.DataFrame(metrics_list)[
    ['model', 'threshold', 'roc_auc', 'pr_auc', 'precision', 'recall', 'f1',
     'true_positives', 'false_positives', 'false_negatives', 'fraud_caught_pct']
]
metrics_df = metrics_df.sort_values('roc_auc', ascending=False)
print(metrics_df.to_string(index=False))


In [ ]:
# ── Confusion Matrices — Best Model ──────────────────────────────────────
best_model_name = metrics_df.iloc[0]['model']
best_prob = probs[best_model_name]
opt_thresh = float(metrics_df.iloc[0]['threshold'])

fig = plot_confusion_matrix(y_test, best_prob, best_model_name,
                             threshold=opt_thresh,
                             save_path='../visuals/06_confusion_matrix.png')


In [ ]:
# ── Feature Importance — LightGBM ─────────────────────────────────────────
lgbm = pickle.load(open('../models/lightgbm.pkl', 'rb'))

fi_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': lgbm.feature_importances_
}).sort_values('Importance', ascending=True).tail(20)

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#E15759' if i > fi_df['Importance'].quantile(0.75) else '#4E79A7'
          for i in fi_df['Importance']]
ax.barh(fi_df['Feature'], fi_df['Importance'], color=colors, edgecolor='white', linewidth=0.3)
ax.set_xlabel('Feature Importance', color='white')
ax.set_title('LightGBM — Top 20 Feature Importances', color='white', fontsize=13)
ax.grid(True, alpha=0.2, axis='x')
plt.tight_layout()
plt.savefig('../visuals/06_feature_importance_lgbm.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()


In [ ]:
# ── SHAP Values (Interpretability) ────────────────────────────────────────
try:
    import shap
    print("Computing SHAP values for LightGBM (sample of 500 test rows)...")
    explainer = shap.TreeExplainer(lgbm)
    X_sample = X_test.sample(500, random_state=42)
    shap_values = explainer.shap_values(X_sample)
    sv = shap_values[1] if isinstance(shap_values, list) else shap_values
    
    # Summary plot
    fig, ax = plt.subplots(figsize=(10, 8))
    shap.summary_plot(sv, X_sample, feature_names=feature_cols,
                      show=False, plot_size=(10, 8))
    plt.title('SHAP Summary Plot — LightGBM', color='white', fontsize=13)
    plt.tight_layout()
    plt.savefig('../visuals/06_shap_summary.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✓ SHAP values computed successfully")
except ImportError:
    print("SHAP not installed. Run: pip install shap")
except Exception as e:
    print(f"SHAP error: {e}")


In [ ]:
# ── Business ROI Analysis ─────────────────────────────────────────────────
print("Business ROI at Optimal Threshold (avg fraud = $856K, investigation cost = $50)\n")
roi_results = []
for name, y_prob in probs.items():
    opt_thresh = find_optimal_threshold(y_test, y_prob, metric='f1')
    roi = compute_business_roi(y_test, y_prob, threshold=opt_thresh,
                               avg_fraud_amount=856_000, investigation_cost=50)
    roi['model'] = name
    roi_results.append(roi)

roi_df = pd.DataFrame(roi_results)[
    ['model', 'fraud_prevented_usd', 'investigation_cost_usd',
     'missed_fraud_loss_usd', 'net_benefit_usd', 'roi_pct']
]
roi_df = roi_df.sort_values('net_benefit_usd', ascending=False)
for col in ['fraud_prevented_usd', 'investigation_cost_usd', 'missed_fraud_loss_usd', 'net_benefit_usd']:
    roi_df[col] = roi_df[col].apply(lambda x: f'${x:,.0f}')
roi_df['roi_pct'] = roi_df['roi_pct'].apply(lambda x: f'{x:.0f}%')
print(roi_df.to_string(index=False))
